# TẬP DỮ LIỆU CV 54K

## Set up các requirement cần thiết để tải dữ liệu từ Kaggle về và xử lý chúng

In [ ]:
# Cài đặt thư viện cần thiết
!pip install opendatasets

In [ ]:
import pandas as pd
import opendatasets as od
import matplotlib.pyplot as plt
import numpy as np
import os, re
import csv
# Tải dữ liệu về
od.download('https://www.kaggle.com/datasets/suriyaganesh/resume-dataset-structured/data')

Please provide your Kaggle credentials to download this dataset. Learn more: http://bit.ly/kaggle-creds
Your Kaggle username: anhkhoi5602
Your Kaggle Key: ··········
Dataset URL: https://www.kaggle.com/datasets/suriyaganesh/resume-dataset-structured


100%|██████████| 38.0M/38.0M [00:00<00:00, 1.07GB/s]

Thêm đường dẫn file vào Python, với mục đích để Python đọc được các file cần thiết

In [ ]:
# Tạo path cho Python
INPUT_PATH = '/BTL_Python/data/raw_data'

# Nếu chưa có folder, thì tạo folder tương ứng
if not os.path.exists(INPUT_PATH):
  os.makedirs(INPUT_PATH)

csv_file = os.listdir(INPUT_PATH)
if csv_file:
    csv_path = [os.path.join(INPUT_PATH,fi) for fi in csv_file]
    print(f"Đường dẫn các Link tương ứng: ")
    for idx,fou_path in enumerate(csv_file,1):
        print(f"{idx}: {fou_path}")

Đường dẫn các Link tương ứng: 
1: 06_skills.csv
2: 04_experience.csv
3: 03_education.csv
4: 01_people.csv
5: 02_abilities.csv
6: 05_person_skills.csv


## Tiến hành đọc file

In [ ]:
# Đọc file .csv các thông tin vào Python bằng Pandas
try:
    skills = pd.read_csv(csv_path[0])
    experience = pd.read_csv(csv_path[1])
    education = pd.read_csv(csv_path[2])
    people = pd.read_csv(csv_path[3])
    abilities = pd.read_csv(csv_path[4])
    person_skills = pd.read_csv(csv_path[5])
except FileNotFoundError as e:
    print(f"Có ít nhất một file không tồn tại: {e}")

print(f"Đã đọc các file thành công~")

Đã đọc các file thành công~


## Preview thông tin về tập dữ liệu ban đầu

In [ ]:
# Preview bảng people
people.head(20)

,person_id,name,email,phone,linkedin
0,1,Database Administrator,NaN,NaN,NaN
1,2,Database Administrator,NaN,NaN,NaN
2,3,Oracle Database Administrator,NaN,NaN,NaN
3,4,Amazon Redshift Administrator and ETL Develope...,NaN,NaN,NaN
4,5,Scrum Master Scrum Master Scrum Master,NaN,NaN,NaN
5,6,Oracle Database Administrator,NaN,NaN,NaN
6,7,Oracle Database Administrator,NaN,NaN,NaN
7,8,Lead Database Administrator/Developer,NaN,NaN,NaN
8,9,"Dominion Diagnostics, LLC",NaN,Available for contact between 9:00 AM and 9:00 PM,NaN
9,10,Oracle Database Administrator,NaN,NaN,http://www.linkedin.com/in/ericnyambabid20760378


In [ ]:
# Số lượng (hàng x cột) của mỗi bảng
# print(f'Unique Skills: {skills.shape}')
print(f'Experience: {experience.shape}')
print(f'Ability: {abilities.shape}')
print(f'People: {people.shape}')
print(f'Skills: {person_skills.shape}')
print(f'Education: {education.shape}')


Unique Skills: (226760, 1)
Experience: (265404, 6)
Ability: (1219473, 2)
People: (54933, 5)
Skills: (2483376, 2)
Education: (75999, 5)


In [ ]:
def merging_resume_data(people, experience, abilities, person_skills, education: pd.DataFrame) -> pd.DataFrame:
    """
    Dùng để gộp lại các bảng DataFrame từ input thành một bảng duy nhất
    Args:
        people: DataFrame về thông tin ứng viên
        experience: DataFrame về thông tin kinh nghiệm làm việc của ứng viên đó
        abilities: DataFrame về những công việc mà ứng viên đó có thể làm được
        skills: DataFrame về những kĩ năng đạt được của ứng viên đó
        education: DataFrame về trình độ học vấn của ứng viên
    Returns:
        merged_df: Bảng DataFrame được gộp lại từ những tham số trên
    """
    try:
          # Nhóm skills với mỗi person
        skills_per_person = person_skills.groupby("person_id")["skill"] \
        .apply(lambda x: ", ".join(x.dropna().astype(str))).reset_index()

          # Nhóm education với mỗi person
        education_per_person = education.groupby("person_id")["program"] \
        .apply(lambda x: ", ".join(x.dropna().astype(str))).reset_index()

          # Nhóm experience với mỗi person
        experience_per_person = experience.groupby("person_id")["title"] \
        .apply(lambda x: ", ".join(x.dropna().astype(str))).reset_index()

          # Nhóm abilities với mỗi person
        abilities_per_person = abilities.groupby("person_id")["ability"] \
        .apply(lambda x: ", ".join(x.dropna().astype(str))).reset_index()

        merged_df = people\
                 .merge(skills_per_person, on="person_id", how="left")\
                 .merge(education_per_person, on="person_id", how="left")\
                 .merge(experience_per_person, on="person_id", how="left")\
                 .merge(abilities_per_person, on="person_id", how="left")
    except FileNotFoundError:
        print("Không tìm thấy file tương ứng!!!")
        return pd.DataFrame()

    return merged_df.set_index('person_id')

In [ ]:
merged_df = merging_resume_data(people, experience, abilities, person_skills, education)


In [ ]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 54933 entries, 1 to 54933
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      54819 non-null  object
 1   email     1593 non-null   object
 2   phone     1833 non-null   object
 3   linkedin  8538 non-null   object
 4   skill     54858 non-null  object
 5   program   48075 non-null  object
 6   title     54933 non-null  object
 7   ability   54930 non-null  object
dtypes: object(8)
memory usage: 3.8+ MB


In [ ]:
def preprocessing_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Dùng để xử lý những dữ liệu bị thiếu, trùng lắp trong DataFrame sau khi được gộp
    Args:
        df: Bảng dữ liệu ban đầu
    Returns:
        preprocessed_df: Bảng DataFrame đã được xử lý dữ liệu bị trùng lắp, bị thiếu
    """
    preprocessed_df = df.copy()
    # 1. Chuyển tên cột về viết chữ thường, nối bằng dấu gạch dưới (Kiểu snake_case)
    preprocessed_df.columns = preprocessed_df.columns.str.lower().str.replace(' ', '_')

    # 2. Xoá các hàng mà nếu một trong bốn thông tin sau đây bị thiếu: title, skill, ability, program
    columns_to_check = ['skill', 'ability', 'program', 'title']
    for col in columns_to_check:
      # Kiểm tra trong các giá trị của columns cần lọc,
      # có giá trị nào để rỗng không, nếu để rỗng thì fill NaN vào
      if col in preprocessed_df.columns:
        preprocessed_df[col] = preprocessed_df[col].replace(r'^\s*$', np.nan, regex=True)
    preprocessed_df = preprocessed_df.dropna(subset=columns_to_check)

    # 3. Các phần thông tin liên lạc, nếu không có hoặc bị trống thì chỉ cần để Unknown
    preprocessed_df['email'] = preprocessed_df['email'].fillna('Unknown')
    preprocessed_df['phone'] = preprocessed_df['phone'].fillna('Unknown')
    preprocessed_df['linkedin'] = preprocessed_df['linkedin'].fillna('Unknown')
    preprocessed_df['name'] = preprocessed_df['name'].fillna('Unknown')

    # 4. Chuẩn hóa nội dung text (chuyển sang chữ thường)
    preprocessed_df['program'] = preprocessed_df['program'].str.lower()
    preprocessed_df['skill'] = preprocessed_df['skill'].str.lower()
    preprocessed_df['ability'] = preprocessed_df['ability'].str.lower()
    preprocessed_df['title'] = preprocessed_df['title'].str.lower()

    # 5. Xóa các hàng trùng lặp (làm ở bước cuối cùng)
    preprocessed_df = preprocessed_df.drop_duplicates(subset=columns_to_check)

    return preprocessed_df


In [ ]:
# Kiểm tra tập dữ liệu sau khi lọc các trường thông tin bị thiếu
resume_df = preprocessing_dataframe(merged_df)
print(resume_df.info())
# Kiểm tra trùng lắp
print(f"Number of duplicate rows: {resume_df.duplicated(subset= ['skill', 'ability', 'program', 'title']).sum()}")

<class 'pandas.core.frame.DataFrame'>
Index: 14566 entries, 1 to 18305
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      14566 non-null  object
 1   email     14566 non-null  object
 2   phone     14566 non-null  object
 3   linkedin  14566 non-null  object
 4   skill     14566 non-null  object
 5   program   14566 non-null  object
 6   title     14566 non-null  object
 7   ability   14566 non-null  object
dtypes: object(8)
memory usage: 1.0+ MB
None
Number of duplicate rows: 0


## Preview dữ liệu sau khi lọc sơ các dữ liệu bị thiếu. Dễ dàng nhận ra rằng trong đây có trùng lặp.

In [ ]:
resume_df['title']

,title
person_id,
1,"database administrator, database administrator"
2,database administrator
3,"oracle database administrator, oracle database..."
4,amazon redshift administrator and etl develope...
6,"oracle database administrator, oracle database..."
...,...
18298,"python developer, python developer, python dev..."
18301,"sr. python developer, sr. python developer, py..."
18302,"sr. python developer, sr. python developer, py..."


In [ ]:
def normalize_text_simple(text:str)->str:
    """
    Dùng để chuẩn hoá chuỗi các thông tin như xoá các thông tin bị trùng lặp
    Parameters:
        text: Chuỗi cần chuẩn hoá
    Return:
        simplified_text: Chuỗi đã được chuẩn hoá
    """
    # 1. Tách chuỗi bằng dấu phẩy (,) hoặc gạch chéo (/)
    parts = re.split(r'[,\/]', text)

    unique_parts = []
    seen = set() # Dùng set để theo dõi các phần đã thấy

    for part in parts:
        # 3. XÓA BỎ khoảng trắng thừa ở đầu/cuối
        cleaned_part = part.strip()

        # 4. Dùng chữ thường để kiểm tra trùng lặp
        lower_part = cleaned_part.lower()

        # 5. Chỉ thêm nếu chuỗi có nội dung và chưa từng xuất hiện
        if cleaned_part and lower_part not in seen:
            seen.add(lower_part)
            unique_parts.append(cleaned_part) # Giữ lại chuỗi gốc (có viết hoa)

    # 6. Nối các phần duy nhất lại bằng ", "
    return ", ".join(unique_parts)

# Áp dụng với các trường thông tin quan trọng
resume_df['title'] = resume_df['title'].apply(normalize_text_simple)
resume_df['skill'] = resume_df['skill'].apply(normalize_text_simple)
resume_df['program'] = resume_df['program'].apply(normalize_text_simple)
resume_df['ability'] = resume_df['ability'].apply(normalize_text_simple)


In [ ]:
resume_df['title'].head(50)

,title
person_id,
1,database administrator
2,database administrator
3,oracle database administrator
4,amazon redshift administrator and etl develope...
6,"oracle database administrator, field service r..."
7,"oracle database administrator, production supp..."
8,"lead database administrator, developer, produc..."
9,"database administrator, database developer, se..."
10,"oracle database administrator, database admini..."


In [ ]:
resume_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14566 entries, 1 to 18305
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   name      14566 non-null  object
 1   email     14566 non-null  object
 2   phone     14566 non-null  object
 3   linkedin  14566 non-null  object
 4   skill     14566 non-null  object
 5   program   14566 non-null  object
 6   title     14566 non-null  object
 7   ability   14566 non-null  object
dtypes: object(8)
memory usage: 1.0+ MB


In [ ]:
# Ghi lại output
resume_df.to_csv('BTL_Python/data/resume_CLEANED.csv', sep=',',index=False, header = True)